In [1]:
dataset = []
with open('CAR FINANCING AGREEMENT.txt', 'r', encoding='utf-8') as file:
    dataset = file.readlines()
    print(f'Loaded {len(dataset)} entries')

Loaded 17 entries


In [3]:
for e in dataset:
    print(e.strip())

BORROWER: Carlos Brown, residing at 1500 Brazil Avenue - Boston, MA, holder of SSN nº 360132173.
LENDER: FINANCIAL BANK OF AMERICA Inc., registered under EIN nº 00-0000000, headquartered at 1000 Main Avenue, New York, NY.
FINANCED AMOUNT: $82,437.00.
CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).
CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the financed asset, with minimum coverage as required by current legislation.
CLAUSE SIX – DEFAULT: Payment delay will result in a penal

In [2]:
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))


In [3]:
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')


Added chunk 1/17 to the database
Added chunk 2/17 to the database
Added chunk 3/17 to the database
Added chunk 4/17 to the database
Added chunk 5/17 to the database
Added chunk 6/17 to the database
Added chunk 7/17 to the database
Added chunk 8/17 to the database
Added chunk 9/17 to the database
Added chunk 10/17 to the database
Added chunk 11/17 to the database
Added chunk 12/17 to the database
Added chunk 13/17 to the database
Added chunk 14/17 to the database
Added chunk 15/17 to the database
Added chunk 16/17 to the database
Added chunk 17/17 to the database


In [4]:
import numpy as np
#run once after the VectorDB is created
chunks = [chunk for chunk, embed in VECTOR_DB]

In [7]:
chunks[:5]

['BORROWER: Carlos Brown, residing at 1500 Brazil Avenue - Boston, MA, holder of SSN nº 360132173.\n',
 'LENDER: FINANCIAL BANK OF AMERICA Inc., registered under EIN nº 00-0000000, headquartered at 1000 Main Avenue, New York, NY.\n',
 'FINANCED AMOUNT: $82,437.00.\n',
 'CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.\n',
 'CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.\n']

In [5]:
#convert vectorDB to a numpy array
chunk_vectors = np.asarray(
    [embed for chunk, embed in VECTOR_DB], dtype=np.float32
)
#normalize the vectors choose the maximum between the norm and a small value to avoid division by zero
chunk_vectors /= np.maximum(
    np.linalg.norm(chunk_vectors, axis = 1, keepdims = True), 1e-10
)

In [9]:
chunk_vectors.shape

(17, 768)

In [6]:
def retrieve(query, top_n = 5):
        query_vectors = np.asarray(ollama.embed(model =EMBEDDING_MODEL, input = query)['embeddings'][0])
        query_vectors /= max(
                    np.linalg.norm(query_vectors), 1e-10
                    )
        similarities = chunk_vectors @ query_vectors
        top_indices = np.argsort(similarities)[-top_n:][::-1]
        return [(chunks[i], float(similarities[i])) for i in top_indices]

In [10]:
input_query = input('Ask me a question: ')# type your question here

In [11]:
input_query

'According to the provided contract text, what payment amounts and due dates does the schedule specify?Quote or summarize only what the document says.'

In [12]:
retrieved_knowledge = retrieve(input_query)

In [13]:
retrieved_knowledge

[('CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.\n',
  0.6306227782857634),
 ('CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).\n',
  0.6204888263633438),
 ('CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.\n',
  0.6158265062604729),
 ('CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.\n',
  0.6125689791323872),
 ('CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.\n',
  0.6107240023382352)]

In [14]:
print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

Retrieved knowledge:
 - (similarity: 0.63) CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.

 - (similarity: 0.62) CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).

 - (similarity: 0.62) CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.

 - (similarity: 0.61) CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.

 - (similarity: 0.61) CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.



In [15]:
context = '\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])

instruction_prompt = f"""You are a careful assistant that answers questions about a contract.

Answer using only the information explicitly stated in the context below. Do not infer missing terms, calculate unstated values, or treat related information as a direct answer.

Rules:
- Answer the question directly and in plain language.
- Cite each factual statement with the clause heading or number that supports it, for example [CLAUSE THREE – PAYMENT].
- If the context does not contain the requested information, say so clearly. Distinguish what the contract does state from what it does not state.
- Do not present interest, penalties, or other related terms as payment amounts or due dates unless the context explicitly identifies them as such.
- If the context is incomplete or ambiguous, explain the limitation rather than guessing.

Context:
{context}
"""

In [16]:
print(context)

 - CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.

 - CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).

 - CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.

 - CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.

 - CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.



In [17]:
print(instruction_prompt)

You are a careful assistant that answers questions about a contract.

Answer using only the information explicitly stated in the context below. Do not infer missing terms, calculate unstated values, or treat related information as a direct answer.

Rules:
- Answer the question directly and in plain language.
- Cite each factual statement with the clause heading or number that supports it, for example [CLAUSE THREE – PAYMENT].
- If the context does not contain the requested information, say so clearly. Distinguish what the contract does state from what it does not state.
- Do not present interest, penalties, or other related terms as payment amounts or due dates unless the context explicitly identifies them as such.
- If the context is incomplete or ambiguous, explain the limitation rather than guessing.

Context:
 - CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.

 - CLAUSE TH

In [18]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)


Chatbot response:
I cannot provide payment amounts or due dates from the contract text. Is there anything else I can help you with?

## Contracts comparison:

In [7]:
with open('CAR FINANCING AGREEMENT - COMPARISON.txt', 'r', encoding='utf-8') as file:
    dataset2 = [line.strip() for line in file.readlines() if line.strip()]
    print(f'Loaded {len(dataset2)} entries')

Loaded 18 entries


In [8]:
for e in dataset2:
    print(e.strip())

BORROWER: Jordan SMITH.
LENDER: Northstar Credit Union, a fictional institution.
VEHICLE: Fictional 2024 Sedan.
FINANCED AMOUNT: $76,500.00.
CLAUSE ONE – PURPOSE: The LENDER agrees to finance the BORROWER's purchase of the vehicle identified above.
CLAUSE TWO – TERM: The loan term is 48 months, beginning on April 15, 2025.
CLAUSE THREE – PAYMENT: The BORROWER shall make 47 monthly installments of $1,971.00 each, followed by a final installment equal to the remaining loan balance and accrued interest. The first installment is due April 15, 2025; each later installment is due on the 15th day of each month, with the final installment due March 15, 2029. The scheduled installment amount is fixed and is not adjusted by the Consumer Price Index (CPI).
CLAUSE FOUR – INTEREST: Interest accrues at a fixed rate of 0.9% per month. The fixed installment amount stated in CLAUSE THREE includes scheduled principal and interest. No additional interest-rate adjustment is specified in this agreement.
CL

In [9]:
LANGUAGE_MODEL_COMP = "qwen2.5:3b"

In [10]:
for i, chunk in enumerate(dataset2):
    add_chunk_to_database(chunk)
    print(f'Add chunk {i+1}/{len(dataset2)} to the database')

Add chunk 1/18 to the database
Add chunk 2/18 to the database
Add chunk 3/18 to the database
Add chunk 4/18 to the database
Add chunk 5/18 to the database
Add chunk 6/18 to the database
Add chunk 7/18 to the database
Add chunk 8/18 to the database
Add chunk 9/18 to the database
Add chunk 10/18 to the database
Add chunk 11/18 to the database
Add chunk 12/18 to the database
Add chunk 13/18 to the database
Add chunk 14/18 to the database
Add chunk 15/18 to the database
Add chunk 16/18 to the database
Add chunk 17/18 to the database
Add chunk 18/18 to the database


In [11]:
chunks2 = [chunk for chunk, embed in VECTOR_DB]

In [12]:
len(VECTOR_DB)

35

In [13]:
chunks2[-5:]

['CLAUSE TEN – TERMINATION: The loan is satisfied when all amounts due have been paid. The agreement does not specify any other termination procedure.',
 'CLAUSE ELEVEN – GOVERNING LAW AND VENUE: This agreement is governed by the laws of the fictional State of Northstar. Any permitted court proceeding shall take place in the fictional County of Sample.',
 'Northstar, April 01, 2025',
 'Jordan SMITH – BORROWER',
 'Representative – NORTHSTAR CREDIT UNION (LENDER)']

In [14]:
#convert vectorDB to a numpy array
chunk_vectors2 = np.asarray(
    [embed for chunk, embed in VECTOR_DB], dtype=np.float32
)
#normalize the vectors choose the maximum between the norm and a small value to avoid division by zero
chunk_vectors2 /= np.maximum(
    np.linalg.norm(chunk_vectors2, axis = 1, keepdims = True), 1e-10
)